In [ ]:
from visualization_utils import TruthData
import torch
from plotly.subplots import make_subplots
import configparser

model = 'llama-2-13b'
config = configparser.ConfigParser()
config.read('config.ini')
layer = eval(config[model]['probe_layer'])
noperiod = eval(config[model]['noperiod'])

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
import os
os.makedirs('figures', exist_ok=True)
print('config layer', layer, 'noperiod', noperiod, 'device', device)

In [ ]:
fig = TruthData.from_datasets(
    ['cities', 'neg_cities'], # datasets to use
    model='llama-2-13b',
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device
).plot(
    dimensions=2, # 3 dimensions also supported
    dim_offset=0, # increase if you want to ignore the first few PCs
    color='label',
    # if you don't want to plot and do PCA on all datasets:
    # plot_datasets = [ list of datasets to plot here ],
    # pca_datasets = [ list of datasets to use for PCA here]
)
fig.write_image('figures/pca_cities_neg_cities.png', scale=2)
print('saved pca_cities_neg_cities.png')

In [ ]:
# visualization of all datasets
datasets = [
    'cities',
    'neg_cities',
    'sp_en_trans',
    'neg_sp_en_trans',
    'larger_than',
    'smaller_than',
    'cities_cities_conj',
    'cities_cities_disj',
    'companies_true_false',
    'common_claim_true_false',
    'counterfact_true_false'
]

td = TruthData.from_datasets(
    datasets,
    model=model,
    layer=layer,
    center=True,
    noperiod=noperiod,
    device=device)

fig = make_subplots(rows=3, cols=4, subplot_titles=datasets, horizontal_spacing=0.05, vertical_spacing=0.05)
for i, dataset in enumerate(datasets):
    for data in td.plot(
        dimensions=2,
        color='label',
        plot_datasets = [dataset],
        pca_datasets = [dataset],
        ).data:
        fig.add_trace(
            data, row=(i // 4) + 1, col=(i % 4) + 1
        )

for row in range(3):
    for col in range(4):
        fig.update_yaxes(
            scaleanchor = f"x{row * 4 + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=1000,
    width=1000,
    showlegend=False,
    title = {
        'text' : 'PCA visualizations of all datasets',
        'font' : {'size' : 30}
    },
    coloraxis_showscale=False,
)
fig.write_image('figures/all_datasets_pca.png', scale=2)
print('saved all_datasets_pca.png')

In [ ]:
pca_datasets = [
    'cities',
    'larger_than',
    'sp_en_trans',
]

plot_datasets = [
    'cities',
    'larger_than',
    'sp_en_trans',
    'counterfact_true_false',
    'cities_cities_disj'
]

fig = make_subplots(rows=len(pca_datasets), cols=len(plot_datasets), 
    vertical_spacing=0.05, horizontal_spacing=0.01,
    )

for col, plot_dataset in enumerate(plot_datasets):
    fig.update_xaxes(title= {
                    'text': plot_dataset,
                    'font' : {'size' : 20}
                    },
                    row=len(pca_datasets), col=col+1
                )
for row, pca_dataset in enumerate(pca_datasets):
    fig.update_yaxes(title= {
                    'text': pca_dataset,
                    'font' : {'size' : 20}
                    },
                    row=row+1, col=1
                )

td = TruthData.from_datasets(plot_datasets, model=model, layer=layer, center=True, noperiod=noperiod, device=device)

for row, pca_dataset in enumerate(pca_datasets):
    for col, plot_dataset in enumerate(plot_datasets):
        subfig = td.plot(
            dimensions = 2,
            plot_datasets = [plot_dataset],
            pca_datasets = [pca_dataset],
            color='label',
        )
        fig.add_trace(subfig.data[0], row=row+1, col=col+1)

for row in range(len(pca_datasets)):
    for col in range(len(plot_datasets)):
        fig.update_yaxes(
            scaleanchor = f"x{row * len(plot_datasets) + col + 1}",
            scaleratio = 1,
            row=row+1,
            col=col+1,
        )

fig.update_coloraxes(
    colorscale = 'Bluered_r'
)

fig.update_layout(
    height=900,
    width=1500,
    coloraxis_showscale=False,
    title = {
        'text': f'Dataset visualizations in various PCA bases',
        'font' : {'size' : 30}
    }
)

fig.write_image('figures/cross_basis.png', scale=2)
print('saved cross_basis.png')
fig.show()

In [ ]:
from plotly.subplots import make_subplots

pairs = [
    ['cities', 'neg_cities'],
    ['sp_en_trans', 'neg_sp_en_trans'],
    ['larger_than', 'smaller_than'],
]
layer = 12

# colormappings under the Rainbow colorscale
RED = 1
BLUE = .17
PURPLE = 0
YELLOW = .73

fig = make_subplots(rows=1, cols=len(pairs),
                    shared_yaxes=True,
                    x_title='PC1', y_title='PC2',
                    subplot_titles=[ '+'.join([dataset for dataset in pair]) for pair in pairs]
                    )

for i, pair in enumerate(pairs):
    td = TruthData.from_datasets(
        pair,
        model='llama-2-13b',
        layer=layer,
        device=device
    )

    
    import numpy as _np
    _lab = td.df['label'].values
    _m0 = td.df.index.get_level_values(0) == pair[0]
    td.df['label'] = _np.where(_m0, _np.where(_lab == 1, BLUE, RED), _np.where(_lab == 1, YELLOW, PURPLE))



    subfig = td.plot(
        dimensions=2,
        color = 'label',
    )

    fig.add_trace(subfig.data[0], row=1, col=i+1)

fig.update_coloraxes(
    colorscale='Rainbow'
)

fig.write_image('figures/negation_pairs_layer12.png', scale=2)
print('saved negation_pairs_layer12.png')
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

datasets = ['sp_en_trans', 'larger_than', 'cities_cities_conj']
layers = [6, 8, 10, 12, 36]

figs = [[] for _ in datasets]

for i, dataset in enumerate(datasets):
    for layer in layers:
        fig = TruthData.from_datasets(
            [dataset],
            model='llama-2-13b',
            layer=layer,
            device=device
            ).plot(
                dimensions=2,
                color='label',
            )
        figs[i].append(fig)

fig = make_subplots(rows = len(datasets), cols = len(layers),
                    subplot_titles=[f"layer {layer}" for layer in layers],
                    vertical_spacing=0.05)

for i, dataset in enumerate(datasets):
    for j, layer in enumerate(layers):
        for data in figs[i][j].data:
            data['showlegend'] = False
            fig.add_trace(data, row=i+1, col=j+1)

for i, dataset in enumerate(datasets):
    fig.update_yaxes(title_text=dataset, row=i+1, col=1)

fig.update_coloraxes(
    colorscale='Bluered_r'
)

fig.update_layout(height=800, width=1200, coloraxis_showscale=False)

fig.update_layout(
    title = {
        'text' : f"Dataset visualizations across layers",
        'font' : {'size' : 20},
    }
)

fig.write_image('figures/layer_sweep.png', scale=2)
print('saved layer_sweep.png')

fig.show()

In [ ]:

# Axis-rotation panel: cities + neg_cities in their own PCA basis at early/mid/late layers.
from plotly.subplots import make_subplots
from visualization_utils import TruthData
RED, BLUE, PURPLE, YELLOW = 1, .17, 0, .73
rot_layers = [4, 12, 24]
rfig = make_subplots(rows=1, cols=len(rot_layers), shared_yaxes=True,
                     x_title='PC1', y_title='PC2',
                     subplot_titles=[f'layer {L}' for L in rot_layers])
for j, L in enumerate(rot_layers):
    import numpy as _np
    td = TruthData.from_datasets(['cities', 'neg_cities'], model='llama-2-13b', layer=L, device=device)
    _lab = td.df['label'].values
    _m0 = td.df.index.get_level_values(0) == 'cities'
    td.df['label'] = _np.where(_m0, _np.where(_lab == 1, BLUE, RED), _np.where(_lab == 1, YELLOW, PURPLE))
    sub = td.plot(dimensions=2, color='label')
    sub.data[0]['showlegend'] = False
    rfig.add_trace(sub.data[0], row=1, col=j+1)
    # also save an individual panel per layer
    sfig = td.plot(dimensions=2, color='label')
    sfig.update_coloraxes(colorscale='Rainbow')
    sfig.write_image(f'figures/rotation_layer_{L}.png', scale=2)
rfig.update_coloraxes(colorscale='Rainbow')
rfig.update_layout(height=450, width=1200, coloraxis_showscale=False,
                   title={'text': 'cities+neg_cities truth-axis rotation across layers', 'font': {'size': 20}})
rfig.write_image('figures/rotation_layers_panel.png', scale=2)
print('saved rotation_layer_{4,12,24}.png and rotation_layers_panel.png')
